STEP 1: Imports

In [1]:
import os
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
import joblib
 


STEP 2: Setup Paths

In [2]:
BASE = r"D:\Deepfake_Audio_Project"
MODEL_PATH = os.path.join(BASE, "models")
 
print(f"\nModel path: {MODEL_PATH}")


Model path: D:\Deepfake_Audio_Project\models


STEP 3: Load DEV Probabilities (for Training Meta-Model)

In [3]:
print("\n" + "="*70)
print("LOADING DEV PROBABILITIES (Meta-Model Training Data)")
print("="*70)
 
# Load CNN probabilities
cnn_dev_probs = np.load(os.path.join(MODEL_PATH, "cnn_dev_probs.npy"))
cnn_dev_probs = np.clip(cnn_dev_probs, 0.05, 0.95)
print(f"✅ CNN DEV probabilities loaded: {cnn_dev_probs.shape}")
 
# Load LGBM probabilities
lgbm_dev_probs = np.load(os.path.join(MODEL_PATH, "lgbm_dev_probs.npy"))
print(f"✅ LGBM DEV probabilities loaded: {lgbm_dev_probs.shape}")
 
# Load true labels
dev_labels = np.load(os.path.join(MODEL_PATH, "dev_labels.npy"))
print(f"✅ DEV labels loaded: {dev_labels.shape}")
 
# Verify shapes match
assert len(cnn_dev_probs) == len(lgbm_dev_probs) == len(dev_labels), "Shape mismatch!"
print(f"\n✅ All DEV data shapes match: {len(dev_labels)} samples")


LOADING DEV PROBABILITIES (Meta-Model Training Data)
✅ CNN DEV probabilities loaded: (24844,)
✅ LGBM DEV probabilities loaded: (24844,)
✅ DEV labels loaded: (24844,)

✅ All DEV data shapes match: 24844 samples


STEP 4: Verify Labels Match (Safety Check)

In [4]:
print("\n" + "="*70)
print("SAFETY CHECK: Verifying Label Consistency")
print("="*70)
 
# Check if CNN and LGBM labels are the same
dev_labels_cnn = np.load(os.path.join(MODEL_PATH, "dev_labels_cnn.npy"))
 
if np.array_equal(dev_labels, dev_labels_cnn):
    print("✅ Labels match perfectly between CNN and LGBM!")
else:
    print("⚠️  WARNING: Label mismatch detected!")
    mismatch_count = np.sum(dev_labels != dev_labels_cnn)
    print(f"   Mismatches: {mismatch_count} out of {len(dev_labels)}")
    print("   This might indicate a sorting issue.")


SAFETY CHECK: Verifying Label Consistency
✅ Labels match perfectly between CNN and LGBM!


STEP 5: Create Meta-Training Dataset

In [5]:
print("\n" + "="*70)
print("CREATING META-TRAINING DATASET")
print("="*70)
 
# Stack CNN and LGBM probabilities side-by-side
from sklearn.preprocessing import StandardScaler

# Stack features
X_meta_dev = np.column_stack([
    cnn_dev_probs,
    1.5 * lgbm_dev_probs   # boost LGBM influence
])

y_meta_dev = dev_labels

# Scale features
scaler = StandardScaler()
X_meta_dev = scaler.fit_transform(X_meta_dev)
 
print(f"Meta-training features shape: {X_meta_dev.shape}")
print(f"  - Column 0: CNN probabilities (confidence that audio is FAKE)")
print(f"  - Column 1: LGBM probabilities (confidence that audio is FAKE)")
print(f"Meta-training labels shape: {y_meta_dev.shape}")
 
# Show example
print(f"\nExample meta-training samples:")
print(f"{'CNN_Prob':<12} {'LGBM_Prob':<12} {'True_Label':<12} {'Interpretation'}")
print("-" * 70)
for i in range(5):
    cnn_p = X_meta_dev[i, 0]
    lgbm_p = X_meta_dev[i, 1]
    label = y_meta_dev[i]
    label_str = "FAKE" if label == 1 else "REAL"
    print(f"{cnn_p:<12.4f} {lgbm_p:<12.4f} {label_str:<12} ", end="")
    
    # Interpretation
    if label == 0:  # Real
        if cnn_p < 0.5 and lgbm_p < 0.5:
            print("Both agree: Real")
        elif cnn_p < 0.5 and lgbm_p > 0.5:
            print("CNN correct, LGBM wrong")
        elif cnn_p > 0.5 and lgbm_p < 0.5:
            print("LGBM correct, CNN wrong")
        else:
            print("Both wrong")
    else:  # Fake
        if cnn_p > 0.5 and lgbm_p > 0.5:
            print("Both agree: Fake")
        elif cnn_p > 0.5 and lgbm_p < 0.5:
            print("CNN correct, LGBM wrong")
        elif cnn_p < 0.5 and lgbm_p > 0.5:
            print("LGBM correct, CNN wrong")
        else:
            print("Both wrong")


CREATING META-TRAINING DATASET
Meta-training features shape: (24844, 2)
  - Column 0: CNN probabilities (confidence that audio is FAKE)
  - Column 1: LGBM probabilities (confidence that audio is FAKE)
Meta-training labels shape: (24844,)

Example meta-training samples:
CNN_Prob     LGBM_Prob    True_Label   Interpretation
----------------------------------------------------------------------
-3.0258      0.0313       REAL         Both agree: Real
-3.0258      -1.4249      REAL         Both agree: Real
-3.0258      0.3130       REAL         Both agree: Real
-3.0258      0.3158       REAL         Both agree: Real
-3.0258      -2.8082      REAL         Both agree: Real


STEP 6: Train Meta-Model (Logistic Regression)

In [6]:
print("\n" + "="*70)
print("TRAINING META-MODEL (Logistic Regression)")
print("="*70)
 
# Create and train meta-model
meta_model = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight={0:1, 1:2},  # give more importance to fake class
    C=0.5
)
 
print("Training meta-model on DEV data...")
meta_model.fit(X_meta_dev, y_meta_dev)
print("✅ Meta-model trained!")
 
# Show what the meta-model learned
print(f"\nMeta-Model Learned Weights:")
print(f"  CNN weight:  {meta_model.coef_[0][0]:.4f}")
print(f"  LGBM weight: {meta_model.coef_[0][1]:.4f}")
print(f"  Intercept:   {meta_model.intercept_[0]:.4f}")
 
if abs(meta_model.coef_[0][0]) > abs(meta_model.coef_[0][1]):
    print("\n💡 Meta-model relies more on CNN predictions")
elif abs(meta_model.coef_[0][1]) > abs(meta_model.coef_[0][0]):
    print("\n💡 Meta-model relies more on LGBM predictions")
else:
    print("\n💡 Meta-model balances both models equally")


TRAINING META-MODEL (Logistic Regression)
Training meta-model on DEV data...
✅ Meta-model trained!

Meta-Model Learned Weights:
  CNN weight:  5.1792
  LGBM weight: 0.6188
  Intercept:   5.0910

💡 Meta-model relies more on CNN predictions


STEP 7: Evaluate on DEV (Sanity Check)

In [7]:
print("\n" + "="*70)
print("DEV PERFORMANCE (Sanity Check - Meta-Model Training Data)")
print("="*70)
 
dev_meta_pred = meta_model.predict(X_meta_dev)
dev_meta_acc = accuracy_score(y_meta_dev, dev_meta_pred)
 
print(f"\nDEV Accuracy: {dev_meta_acc:.4f}")
print("\nDEV Classification Report:")
print(classification_report(y_meta_dev, dev_meta_pred, target_names=['Real', 'Fake']))
 
print("\nDEV Confusion Matrix:")
cm_dev = confusion_matrix(y_meta_dev, dev_meta_pred)
print(cm_dev)


DEV PERFORMANCE (Sanity Check - Meta-Model Training Data)

DEV Accuracy: 0.9977

DEV Classification Report:
              precision    recall  f1-score   support

        Real       1.00      0.98      0.99      2548
        Fake       1.00      1.00      1.00     22296

    accuracy                           1.00     24844
   macro avg       1.00      0.99      0.99     24844
weighted avg       1.00      1.00      1.00     24844


DEV Confusion Matrix:
[[ 2494    54]
 [    2 22294]]


STEP 8: Load EVAL Probabilities (for Testing)

In [8]:
print("\n" + "="*70)
print("LOADING EVAL PROBABILITIES (Final Test)")
print("="*70)
 
cnn_eval_probs = np.load(os.path.join(MODEL_PATH, "cnn_eval_probs.npy"))
cnn_eval_probs = np.clip(cnn_eval_probs, 0.05, 0.95)
print(f"✅ CNN EVAL probabilities loaded: {cnn_eval_probs.shape}")
 
lgbm_eval_probs = np.load(os.path.join(MODEL_PATH, "lgbm_eval_probs.npy"))
print(f"✅ LGBM EVAL probabilities loaded: {lgbm_eval_probs.shape}")
 
eval_labels = np.load(os.path.join(MODEL_PATH, "eval_labels.npy"))
print(f"✅ EVAL labels loaded: {eval_labels.shape}")
 
# Create meta-test dataset
X_meta_eval = np.column_stack([
    cnn_eval_probs,
    1.5 * lgbm_eval_probs
])
X_meta_eval = scaler.transform(X_meta_eval)
y_meta_eval = eval_labels
 
print(f"\nMeta-test features shape: {X_meta_eval.shape}")


LOADING EVAL PROBABILITIES (Final Test)
✅ CNN EVAL probabilities loaded: (71237,)
✅ LGBM EVAL probabilities loaded: (71237,)
✅ EVAL labels loaded: (71237,)

Meta-test features shape: (71237, 2)


STEP 9: Final Evaluation on EVAL

In [9]:
print("\n" + "="*70)
print("🎯 FINAL SYSTEM EVALUATION (EVAL Dataset)")
print("="*70)
 
eval_meta_probs = meta_model.predict_proba(X_meta_eval)[:,1]

# Adjust threshold (try 0.6–0.7 range)
threshold = 0.6
eval_meta_pred = (eval_meta_probs > threshold).astype(int)


eval_meta_acc = accuracy_score(y_meta_eval, eval_meta_pred)
 
# Calculate ROC AUC
eval_meta_probs_for_auc = meta_model.predict_proba(X_meta_eval)[:, 1]
eval_roc_auc = roc_auc_score(y_meta_eval, eval_meta_probs_for_auc)
 
print(f"\n{'='*70}")
print(f"🎉 FINAL SYSTEM ACCURACY: {eval_meta_acc:.4f} ({eval_meta_acc*100:.2f}%)")
print(f"🎯 ROC AUC SCORE: {eval_roc_auc:.4f}")
print(f"{'='*70}")
 
print("\n📊 EVAL Classification Report:")
print(classification_report(y_meta_eval, eval_meta_pred, target_names=['Real', 'Fake']))
 
print("\n📊 EVAL Confusion Matrix:")
cm_eval = confusion_matrix(y_meta_eval, eval_meta_pred)
print(cm_eval)
print("Format: [[True_Real, False_Fake],")
print("         [False_Real, True_Fake]]")
 
# Detailed metrics
tn, fp, fn, tp = cm_eval.ravel()
real_recall = tn / (tn + fp) if (tn + fp) > 0 else 0
fake_recall = tp / (tp + fn) if (tp + fn) > 0 else 0
real_precision = tn / (tn + fn) if (tn + fn) > 0 else 0
fake_precision = tp / (tp + fp) if (tp + fp) > 0 else 0
 
# Calculate actual counts
real_count = np.sum(y_meta_eval == 0)
fake_count = np.sum(y_meta_eval == 1)
 
print(f"\n📈 Detailed EVAL Metrics:")
print(f"Real samples ({real_count}):")
print(f"  Correctly detected: {tn} ({tn/real_count*100:.1f}%)")
print(f"  Missed (False Fake): {fp} ({fp/real_count*100:.1f}%)")
print(f"  Recall: {real_recall:.4f}")
print(f"  Precision: {real_precision:.4f}")
 
print(f"\nFake samples ({fake_count}):")
print(f"  Correctly detected: {tp} ({tp/fake_count*100:.1f}%)")
print(f"  Missed (False Real): {fn} ({fn/fake_count*100:.1f}%)")
print(f"  Recall: {fake_recall:.4f}")
print(f"  Precision: {fake_precision:.4f}")


🎯 FINAL SYSTEM EVALUATION (EVAL Dataset)

🎉 FINAL SYSTEM ACCURACY: 0.8748 (87.48%)
🎯 ROC AUC SCORE: 0.9537

📊 EVAL Classification Report:
              precision    recall  f1-score   support

        Real       0.45      0.98      0.62      7355
        Fake       1.00      0.86      0.93     63882

    accuracy                           0.87     71237
   macro avg       0.72      0.92      0.77     71237
weighted avg       0.94      0.87      0.89     71237


📊 EVAL Confusion Matrix:
[[ 7190   165]
 [ 8754 55128]]
Format: [[True_Real, False_Fake],
         [False_Real, True_Fake]]

📈 Detailed EVAL Metrics:
Real samples (7355):
  Correctly detected: 7190 (97.8%)
  Missed (False Fake): 165 (2.2%)
  Recall: 0.9776
  Precision: 0.4510

Fake samples (63882):
  Correctly detected: 55128 (86.3%)
  Missed (False Real): 8754 (13.7%)
  Recall: 0.8630
  Precision: 0.9970


STEP 10: Compare Individual Models vs Ensemble

In [10]:
print("\n" + "="*70)
print("📊 PERFORMANCE COMPARISON: Individual Models vs Ensemble")
print("="*70)
 
# CNN alone on EVAL
cnn_eval_pred = (cnn_eval_probs > 0.5).astype(int)
cnn_eval_acc = accuracy_score(y_meta_eval, cnn_eval_pred)
 
# LGBM alone on EVAL
lgbm_eval_pred = (lgbm_eval_probs > 0.5).astype(int)
lgbm_eval_acc = accuracy_score(y_meta_eval, lgbm_eval_pred)
 
print(f"\n{'Model':<20} {'Accuracy':<12} {'Real Recall':<15} {'Fake Recall'}")
print("-" * 70)
 
# CNN metrics
cnn_cm = confusion_matrix(y_meta_eval, cnn_eval_pred)
cnn_tn, cnn_fp, cnn_fn, cnn_tp = cnn_cm.ravel()
cnn_real_recall = cnn_tn / (cnn_tn + cnn_fp)
cnn_fake_recall = cnn_tp / (cnn_tp + cnn_fn)
print(f"{'CNN (alone)':<20} {cnn_eval_acc:<12.4f} {cnn_real_recall:<15.4f} {cnn_fake_recall:.4f}")
 
# LGBM metrics
lgbm_cm = confusion_matrix(y_meta_eval, lgbm_eval_pred)
lgbm_tn, lgbm_fp, lgbm_fn, lgbm_tp = lgbm_cm.ravel()
lgbm_real_recall = lgbm_tn / (lgbm_tn + lgbm_fp)
lgbm_fake_recall = lgbm_tp / (lgbm_tp + lgbm_fn)
print(f"{'LightGBM (alone)':<20} {lgbm_eval_acc:<12.4f} {lgbm_real_recall:<15.4f} {lgbm_fake_recall:.4f}")
 
# Ensemble
print(f"{'Ensemble (Meta)':<20} {eval_meta_acc:<12.4f} {real_recall:<15.4f} {fake_recall:.4f}")
 
print(f"\n💡 Improvement over best individual model:")
best_individual = max(cnn_eval_acc, lgbm_eval_acc)
improvement = ((eval_meta_acc - best_individual) / best_individual) * 100
print(f"   {improvement:+.2f}% accuracy gain!")


📊 PERFORMANCE COMPARISON: Individual Models vs Ensemble

Model                Accuracy     Real Recall     Fake Recall
----------------------------------------------------------------------
CNN (alone)          0.8809       0.9742          0.8702
LightGBM (alone)     0.8865       0.5453          0.9258
Ensemble (Meta)      0.8748       0.9776          0.8630

💡 Improvement over best individual model:
   -1.32% accuracy gain!


STEP 11: Save Meta-Model

In [11]:
print("\n" + "="*70)
print("SAVING META-MODEL")
print("="*70)
 
meta_model_path = os.path.join(MODEL_PATH, "meta_model.pkl")
joblib.dump(meta_model, meta_model_path)
print(f"✅ Meta-model saved to: {meta_model_path}")


SAVING META-MODEL
✅ Meta-model saved to: D:\Deepfake_Audio_Project\models\meta_model.pkl


STEP 12: Save Final Predictions

In [12]:
print("\n" + "="*70)
print("SAVING FINAL PREDICTIONS")
print("="*70)
 
# Save EVAL predictions
np.save(os.path.join(MODEL_PATH, "final_eval_predictions.npy"), eval_meta_pred)
print("✅ EVAL predictions saved")
 
# Save prediction probabilities
eval_meta_probs = meta_model.predict_proba(X_meta_eval)[:, 1]
np.save(os.path.join(MODEL_PATH, "final_eval_probabilities.npy"), eval_meta_probs)
print("✅ EVAL probabilities saved")
 
# ===== SUMMARY =====
print("\n" + "="*70)
print("🎉 META-MODEL TRAINING COMPLETE!")
print("="*70)
 
print(f"\n📁 Files saved in: {MODEL_PATH}")
print("  - meta_model.pkl (trained ensemble model)")
print("  - final_eval_predictions.npy (predictions on EVAL)")
print("  - final_eval_probabilities.npy (confidence scores)")
 
print(f"\n📊 Final System Performance:")
print(f"  Overall Accuracy: {eval_meta_acc*100:.2f}%")
print(f"  Real Detection: {real_recall*100:.1f}%")
print(f"  Fake Detection: {fake_recall*100:.1f}%")
 
print("\n✅ Ready for Stage 6: Flask Application!")
print("="*70)


SAVING FINAL PREDICTIONS
✅ EVAL predictions saved
✅ EVAL probabilities saved

🎉 META-MODEL TRAINING COMPLETE!

📁 Files saved in: D:\Deepfake_Audio_Project\models
  - meta_model.pkl (trained ensemble model)
  - final_eval_predictions.npy (predictions on EVAL)
  - final_eval_probabilities.npy (confidence scores)

📊 Final System Performance:
  Overall Accuracy: 87.48%
  Real Detection: 97.8%
  Fake Detection: 86.3%

✅ Ready for Stage 6: Flask Application!
